<a href="https://colab.research.google.com/github/YaninaColangelo/market_labor_analysis/blob/main/notebooks/00_1_exploracion_fuente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/YaninaColangelo/market_labor_analysis/blob/main/notebooks/00_1_exploracion_fuente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00_1 - Exploracion de fuente

Objetivo: comprobar que una oferta publicada por URL puede extraerse, transformarse al esquema `data/raw/ofertas_brutas.csv` y recorrer el ciclo de limpieza y clasificacion del repositorio.

# 0 - Preparacion del entorno

In [1]:
!git clone https://github.com/YaninaColangelo/market_labor_analysis.git


Cloning into 'market_labor_analysis'...
remote: Enumerating objects: 137, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 137 (delta 66), reused 92 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (137/137), 75.22 KiB | 3.58 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [2]:
%cd market_labor_analysis

/content/market_labor_analysis


In [3]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

In [4]:
from src.connectors.tecnoempleo import obtener_oferta
from src.etl import transformar_oferta
from src.data_collection import (
    append_offers,
    load_raw_dataset,
    normalize_offer_to_raw_schema,
    save_raw_dataset,
)
from src.data_cleaning import run_cleaning
from src.data_classification import run_classification

# 1 - Extract

Se prueba una URL concreta. En local Tecnoempleo puede bloquear la descarga; esta celda esta pensada para ejecutarse en Colab.

In [5]:
url = "https://www.tecnoempleo.com/analista-datos-powerbi-sopra-steria/power-bi-etl/rf-7624176cc2dbe32ca545"

oferta_extraida = obtener_oferta(url)
oferta_extraida

{'id_oferta': 'TEC-004b65acd1a5',
 'fecha_publicacion': '2026-06-30',
 'fuente': 'Tecnoempleo',
 'url': 'https://www.tecnoempleo.com/analista-datos-powerbi-sopra-steria/power-bi-etl/rf-7624176cc2dbe32ca545',
 'pais': 'ES',
 'titulo': 'Analista de datos PowerBI',
 'descripcion': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'responsabilidades': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'requisitos': '- Experiencia de, al menos, 2 años con Power BI - Experiencia con procesos ETL - Experiencia en elaboración de dashboard/informes - Experiencia con bbdd SQL Server - Inglés alto C1',
 'ciudad': 'Madrid',
 'empresa': 'Sopra Steria',
 'industria': 'sector bancario',
 'modalidad': 'P

# 2 - Transform

La transformacion queda centralizada en `src.etl.transformar_oferta`: separa titulo/ciudad, normaliza texto y completa campos disponibles desde la metadata.

In [6]:
oferta_transformada = transformar_oferta(oferta_extraida)
oferta_transformada

{'id_oferta': 'TEC-004b65acd1a5',
 'fecha_publicacion': '2026-06-30',
 'fuente': 'Tecnoempleo',
 'url': 'https://www.tecnoempleo.com/analista-datos-powerbi-sopra-steria/power-bi-etl/rf-7624176cc2dbe32ca545',
 'pais': 'ES',
 'ciudad': 'Madrid',
 'empresa': 'Sopra Steria',
 'industria': 'sector bancario',
 'titulo_puesto': 'Analista de datos PowerBI',
 'descripcion': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'responsabilidades': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'requisitos': '- Experiencia de, al menos, 2 años con Power BI - Experiencia con procesos ETL - Experiencia en elaboración de dashboard/informes - Experiencia con bbdd SQL Server - Inglés alto C1',
 'modalid

In [7]:
campos_requeridos = [
    "fecha_publicacion",
    "empresa",
    "industria",
    "modalidad",
    "seniority",
    "formacion_requerida",
]

{k: oferta_extraida.get(k) for k in campos_requeridos}

{'fecha_publicacion': '2026-06-30',
 'empresa': 'Sopra Steria',
 'industria': 'sector bancario',
 'modalidad': 'Presencial',
 'seniority': '2 años',
 'formacion_requerida': ''}

# 3 - Adaptacion al raw dataset

La oferta se ajusta a las columnas de `data/raw/ofertas_brutas.csv`. El `id_oferta` queda vacio para que `data_collection` asigne el siguiente codigo incremental `OF-...`.

In [8]:
oferta_raw = normalize_offer_to_raw_schema(oferta_transformada)
oferta_raw["id_oferta"] = None

oferta_raw

{'id_oferta': None,
 'fecha_publicacion': '2026-06-30',
 'fuente': 'Tecnoempleo',
 'url': 'https://www.tecnoempleo.com/analista-datos-powerbi-sopra-steria/power-bi-etl/rf-7624176cc2dbe32ca545',
 'pais': 'ES',
 'ciudad': 'Madrid',
 'empresa': 'Sopra Steria',
 'industria': 'sector bancario',
 'titulo_puesto': 'Analista de datos PowerBI',
 'descripcion': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'responsabilidades': 'Buscamos un perfil de Análisis de datos con C1 de inglés y experiencia con Power BI para incorporarse en un proyecto estable del sector bancario. Presencialidad flexible Horario de oficina flexible',
 'requisitos': '- Experiencia de, al menos, 2 años con Power BI - Experiencia con procesos ETL - Experiencia en elaboración de dashboard/informes - Experiencia con bbdd SQL Server - Inglés alto C1',
 'modalidad': 'Presenci

# 4 - Load

Se carga el dataset raw existente, se reemplaza la misma URL si ya estaba cargada y se guarda el CSV raw.

In [9]:
df_raw = load_raw_dataset()

df_raw = df_raw[df_raw["url"] != oferta_raw["url"]].copy()
df_raw = append_offers(df_raw, [oferta_raw])
save_raw_dataset(df_raw)

df_raw[df_raw["url"] == url].T

,4
id_oferta,OF-005
fecha_publicacion,2026-06-30
fuente,Tecnoempleo
url,https://www.tecnoempleo.com/analista-datos-pow...
pais,ES
ciudad,Madrid
empresa,Sopra Steria
industria,sector bancario
titulo_puesto,Analista de datos PowerBI
descripcion,Buscamos un perfil de Análisis de datos con C1...


# 5 - Validacion del ciclo

Se ejecutan los pasos del repositorio que consumen `ofertas_brutas.csv`: limpieza y clasificacion.

In [10]:
df_limpias = run_cleaning()

df_limpias[df_limpias["url"] == url].T

,4
id_oferta,OF-005
fecha_publicacion,2026-06-30 00:00:00
fuente,Tecnoempleo
url,https://www.tecnoempleo.com/analista-datos-pow...
pais,ES
ciudad,Madrid
empresa,Sopra Steria
industria,sector bancario
titulo_puesto,Analista de datos PowerBI
descripcion,Buscamos un perfil de Análisis de datos con C1...


In [11]:
df_clasificadas = run_classification()

df_clasificadas[df_clasificadas["url"] == url].T

,4
id_oferta,OF-005
fecha_publicacion,2026-06-30
fuente,Tecnoempleo
url,https://www.tecnoempleo.com/analista-datos-pow...
pais,ES
ciudad,Madrid
empresa,Sopra Steria
industria,sector bancario
titulo_puesto,Analista de datos PowerBI
descripcion,Buscamos un perfil de Análisis de datos con C1...


In [12]:
fila = df_clasificadas[df_clasificadas["url"] == url].iloc[0]

fila[campos_requeridos]

,4
fecha_publicacion,2026-06-30
empresa,Sopra Steria
industria,sector bancario
modalidad,Presencial
seniority,2 años
formacion_requerida,NaN


In [13]:
fila[campos_requeridos].isna()[fila[campos_requeridos].isna()]

,4
formacion_requerida,True


## Resultado esperado

Si la pagina entrega esos datos en el HTML, la oferta queda cargada con el esquema raw, ID incremental, texto de analisis y variables de clasificacion. Si algun campo queda vacio, significa que la fuente no lo expuso de forma recuperable para ese caso y no que el notebook haya cambiado el esquema.

In [14]:
import pandas as pd

from src.etl import transformar_oferta
from src.data_collection import (
    append_offers,
    load_raw_dataset,
    normalize_offer_to_raw_schema,
    save_raw_dataset,
)
from src.utils import DATA_RAW, PROJECT_ROOT


# Verificar que estamos trabajando sobre el repo correcto
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW CSV:", DATA_RAW / "ofertas_brutas.csv")


# Recalcular la transformacion desde la oferta extraida actual
oferta_transformada = transformar_oferta(oferta_extraida)

oferta_raw = normalize_offer_to_raw_schema(oferta_transformada)

# Forzar id incremental OF-...
oferta_raw["id_oferta"] = None


# Cargar raw actual
df_raw = load_raw_dataset()

# Reemplazar explicitamente SOLO esta oferta por URL
df_raw = df_raw[df_raw["url"] != oferta_raw["url"]].copy()

# Agregar version actualizada
df_raw = append_offers(df_raw, [oferta_raw])

# Guardar en data/raw/ofertas_brutas.csv
save_raw_dataset(df_raw)


# Leer de nuevo desde disco para comprobar que realmente quedo guardada
df_raw_check = pd.read_csv(DATA_RAW / "ofertas_brutas.csv")

df_raw_check[df_raw_check["url"] == url].T

PROJECT_ROOT: /content/market_labor_analysis
RAW CSV: /content/market_labor_analysis/data/raw/ofertas_brutas.csv


,4
id_oferta,OF-005
fecha_publicacion,2026-06-30
fuente,Tecnoempleo
url,https://www.tecnoempleo.com/analista-datos-pow...
pais,ES
ciudad,Madrid
empresa,Sopra Steria
industria,sector bancario
titulo_puesto,Analista de datos PowerBI
descripcion,Buscamos un perfil de Análisis de datos con C1...


In [15]:
campos_revision = [
    "id_oferta",
    "fecha_publicacion",
    "fuente",
    "url",
    "pais",
    "ciudad",
    "empresa",
    "industria",
    "titulo_puesto",
    "descripcion",
    "responsabilidades",
    "requisitos",
    "modalidad",
    "seniority",
    "formacion_requerida",
]

df_raw_check[df_raw_check["url"] == url][campos_revision].T

,4
id_oferta,OF-005
fecha_publicacion,2026-06-30
fuente,Tecnoempleo
url,https://www.tecnoempleo.com/analista-datos-pow...
pais,ES
ciudad,Madrid
empresa,Sopra Steria
industria,sector bancario
titulo_puesto,Analista de datos PowerBI
descripcion,Buscamos un perfil de Análisis de datos con C1...
